# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Anthony Isaac Chuya Ruiz  <br>
Url: https://github.com/Anthony5432/03MAIR---Algoritmos-de-Optimizacion/tree/main/TRABAJO_PRACTICO<br>
Google Colab: https://colab.research.google.com/drive/1EyAchs7tlboewSg3oMLKyHBlS9IhDVfk?usp=sharing <br>
Problema:

>Organizar los horarios de partidos de una jornada de La Liga<br>

Descripción del problema:

La Liga de Fútbol Profesional quiere asignar los horarios de los partidos de cada jornada de manera que se maximice la audiencia total. Conociendo los equipos que jugarán, su audiencia base, cuanto se verá afectada si es emitido en un horario específico y también si coincide con otro. Y teniendo en cuenta que siempre habrá un partido el viernes y lunes.







                                        

#Modelo
- ¿Como represento el espacio de soluciones?

  El espacio de soluciones está compuesto por todas las asignaciones posibles de partidos a horarios, permitiendo asignar mas de un partido a un mismo horario con penalización y teniendo los partidos del lunes y viernes siempre asignados.

- ¿Cual es la función objetivo?
  El objetivo de este problema es maximizar la audiencia total. Y esta dependen los siguientes factores:
  - Audiencia base (A_{ij}): Depende de los equipos que juegan.
  -Coeficiente horario (C_h) → La audiencia varía  según el horario emitido.
  -Coeficiente de coincidencia (C_c) → Si hay mas de un partido a la vez, la audiencia disminuira para ambos.

  Con lo que la función objetivo quedaría así:
    - max ∑_i (A_{ij}* C_h *C_c)

- ¿Como implemento las restricciones?

  1. Un partido solo y debe asignarse a un horario disponible.
   - Esto se realizo de forma que en el bucle si se asigna un partido se saque de el mismo.
  2. Se debe asignar obligatoriamente un partido el viernes y otro el lunes.
   - Al principio del bucle hay dos horarios obligatorios , esos mismos, lo cuales siempre se asignan primeros, y como son los de menor audiencia posible, se asignan partidos con menor audiencia también para que no afecta a la cantidad posible de audiencia que podría traer mejores partidos.
  3. La audiencia se reduce si varios partidos coinciden en el mismo horario.
   -  Si varios partidos comparten el mismo horario, se aplicará una penalización ya establecidas.

In [ ]:
import numpy as np
import pandas as pd
import math

# Equipos y categorías
equipos = {
    "A1": "A", "A2": "A", "A3": "A",
    "B1": "B", "B2": "B", "B3": "B", "B4": "B", "B5": "B", "B6": "B", "B7": "B", "B8": "B", "B9": "B", "B10": "B", "B11": "B",
    "C1": "C", "C2": "C", "C3": "C", "C4": "C", "C5": "C", "C6": "C"
}
# Se asignan de esta forma los equipos para acceder facilmente a sus categorías

# Horarios y coeficientes
horarios = ["Viernes 20", "Sábado 12", "Sábado 16", "Sábado 18", "Sábado 20", "Domingo 12", "Domingo 16", "Domingo 18", "Domingo 20", "Lunes 20"]
coeficientes_horarios = {
    "Viernes 20": 0.4, "Sábado 12": 0.55, "Sábado 16": 0.7, "Sábado 18": 0.8, "Sábado 20": 1.0,
    "Domingo 12": 0.45, "Domingo 16": 0.75, "Domingo 18": 0.85, "Domingo 20": 1.0, "Lunes 20": 0.4
}

# Audiencia basada en categorías
audiencia_categorias = {
    ("A", "A"): 2.0, ("A", "B"): 1.3, ("A", "C"): 1.0,
    ("B", "A"): 1.3, ("B", "B"): 0.9, ("B", "C"): 0.75,
    ("C", "A"): 1.0, ("C", "B"): 0.75, ("C", "C"): 0.47
}

# Coeficientes de reducción de audiencia por coincidencias
tabla_coincidencias = {1: 1.0, 2: 0.75, 3: 0.55, 4: 0.40, 5: 0.30, 6: 0.25, 7: 0.22, 8: 0.20, 9: 0.2}


In [ ]:


# Se ordena la lista de equipos por categoría (A primero, luego B y C al final)
# Con lo que el primero siempre será uno de los mayores de audiencia
# Esto para una supuesta entrada de equipos que no este ordenado
equipos_ordenados = sorted(equipos.keys(), key=lambda e: equipos[e])


# Tabla de resultados
resultados = []
resultado_global = 0

# Asignación de partidos a horarios, los cuales cuentan cuantos hay en cada uno
disponibilidad_horarios = {h: 0 for h in horarios}  # Contador de partidos por horario

horarios_obligatorios = ["Viernes 20", "Lunes 20"]  # Restricción

partidos_asignados = []  # Lista para almacenar los partidos asignados

# Primera pasada: Asignación sin calcular penalización
while len(equipos_ordenados) > 1:

    if horarios_obligatorios: # Si el lunes y viernes no tiene partido
        equipo1 = equipos_ordenados.pop(-1)  # Elegimos el peor equipo
        mejor_pareja = equipos_ordenados.pop(-1)  # Elegimos el segundo peor equipo
        mejor_horario = horarios_obligatorios.pop(0)  # Marcamos el horario como cubierto
    else:

      # Revisamos primarente los equipos con mayor audiencia
      equipo1 = equipos_ordenados.pop(0)  # Seleccionamos el primer equipo(mejor)
      mejor_pareja = equipos_ordenados.pop(0)  # Seleccionamos el siguiente equipo como la mejor pareja
      mejor_horario = None

      mejor_valor = float("-inf")

      # Buscamos los horarios actuales con mayor posibilidad de audiencia, teniendo en cuenta coincidencias
      for h in horarios:
        valor = coeficientes_horarios[h] * tabla_coincidencias[disponibilidad_horarios[h] + 1]

        if valor > mejor_valor:  # Si encontramos un mejor horario
          mejor_valor = valor
          mejor_horario = h  # Guardamos el mejor horario encontrado

    # Incrementamos la cantidad de partidos en ese horario
    disponibilidad_horarios[mejor_horario] += 1

    # Guardamos el partido sin calcular aún la audiencia final
    partidos_asignados.append((equipo1, mejor_pareja, mejor_horario))

# Segunda pasada: Cálculo de audiencia con penalización correcta
resultado_global = 0
resultados = []

for equipo1, mejor_pareja, horario in partidos_asignados:
    audiencia_base = audiencia_categorias[(equipos[equipo1], equipos[mejor_pareja])]
    coef_horario = coeficientes_horarios[horario]

    # Aquí ya tenemos el número total de partidos por horario, usamos la penalización correcta
    penalizacion = tabla_coincidencias[disponibilidad_horarios[horario]]

    audiencia_final = audiencia_base * coef_horario * penalizacion
    resultado_global += audiencia_final

    resultados.append([
        f"{equipo1} vs {mejor_pareja}", f"{equipos[equipo1]} - {equipos[mejor_pareja]}", horario,
        f"{audiencia_base:.2f} M", coef_horario, penalizacion,
        f"{audiencia_base} * {coef_horario} * {penalizacion}", f"{audiencia_final:.2f} M"
    ])


# Usamos un  DataFrame para la visualización
df_resultados = pd.DataFrame(resultados, columns=[
    "Partido", "Categorías", "Horario",
    "Audiencia Base", "Ponderación Horario", "Penalización Coincidencia",
    "Cálculo", "Audiencia Final"
])



In [ ]:
#Mostramos la tabla de asignaciones de forma visual
df_resultados.style \
  .format(precision=2, thousands=".", decimal=",") \
  .format_index(str.upper, axis=1) \
  .relabel_index(["1", "2","3", "4","5","6","7","8","9","10"], axis=0)

,PARTIDO,CATEGORÍAS,HORARIO,AUDIENCIA BASE,PONDERACIÓN HORARIO,PENALIZACIÓN COINCIDENCIA,CÁLCULO,AUDIENCIA FINAL
1,C6 vs C5,C - C,Viernes 20,0.47 M,"0,40","1,00",0.47 * 0.4 * 1.0,0.19 M
2,C4 vs C3,C - C,Lunes 20,0.47 M,"0,40","1,00",0.47 * 0.4 * 1.0,0.19 M
3,A1 vs A2,A - A,Sábado 20,2.00 M,"1,00","0,75",2.0 * 1.0 * 0.75,1.50 M
4,A3 vs B1,A - B,Domingo 20,1.30 M,"1,00","0,75",1.3 * 1.0 * 0.75,0.98 M
5,B2 vs B3,B - B,Domingo 18,0.90 M,"0,85","1,00",0.9 * 0.85 * 1.0,0.77 M
6,B4 vs B5,B - B,Sábado 18,0.90 M,"0,80","1,00",0.9 * 0.8 * 1.0,0.72 M
7,B6 vs B7,B - B,Sábado 20,0.90 M,"1,00","0,75",0.9 * 1.0 * 0.75,0.68 M
8,B8 vs B9,B - B,Domingo 16,0.90 M,"0,75","1,00",0.9 * 0.75 * 1.0,0.68 M
9,B10 vs B11,B - B,Domingo 20,0.90 M,"1,00","0,75",0.9 * 1.0 * 0.75,0.68 M
10,C1 vs C2,C - C,Sábado 16,0.47 M,"0,70","1,00",0.47 * 0.7 * 1.0,0.33 M


In [ ]:

# Mostrar el total de forma más destacada
print("\n" + "="*60)
print(f"{' '*10}{'Audiencia Total':^40}{' '*10}")
print("="*60)
print(f"{' '*10}{f'{resultado_global:,.2f} M':^40}{' '*10}")
print("="*60)


                      Audiencia Total                       
                           6.69 M                           


#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

- ## Orden de Complejidad
Analizamos el flujo del algoritmo, tomando n como el número de equipos:
1. **Ordenación de los equipos** por categoría, su complejidad es  **O(n*log(n))**, casi lineal.
2. **Primer bucle**, recorre todos los equipos , y como van de 2 en 2, hay n/2 iteraciones o **O(n)**. Y dentro de cada una se busca entre los horarios el mejor posible, teniendo una complejidad de **O(m)**, siendo m el número de horarios. Lo que nos deja una complejidad de  **O(n*m)**
3. **Segundo bucle**, se recorren los partidos asignados para calcular las audiencias. El número de partidos asignados es n/2, por lo que la complejidad del bucle es **O(n)**
4. **Construcción de la tabla**, es una operación lineal respecto al número de partidos asignados, con lo que tendríamos una complejidad de **O(n)**

 Y sumando todos las complejidades:
  **O(n*log(n) + n*m + 2n)**

 Suponiendo que m puede variar y pueda ser grande la complejidad del algoritmo es :  **O(n*m)**

 En caso contrario : **O(n*log(n))**

- ## Espacio de soluciones
 El espacio de soluciones para este problema son todas las posibles asignaciones de partidos a horarios disponibles, considerando las restricciones.

 Este espacio está determinado por el numeró de equipos**(n)**, partidos**(p)**, p = 2/n, y horarios disponibles**(m)**

 Sin considerar la restriccion del lunes y viernes, tendríamos el espacio de soluciones siguiente: **m^(p)**.

 Tomando en cuenta las restricciones, deberíamos no contar la soluciones donde el lunes o viernes o ambas no tengan ningún partido asignado.

 con lo que la resta quedaría :  **m^p - 2*(m-1)^p +(m-2)^p**

 Se restan las combinaciones en las que al menos uno de los 2 este sin partido, y se añaden las que ambos esten vacios una vez, ya que fueron restadas 2 veces.

  Y como tenemos 10 partidos(20 equipos) y 10 horarios, hay **4,100,173,022** soluciones

#Diseño
- ¿Que técnica utilizo? ¿Por qué?

El algoritmo  que utilizo es voraz(o greedy). Esta técnica toma decisiones locales óptimas en cada paso, eligiendo la opción que parece ser la mejor en ese momento sin considerar todas las posibles opciones a largo plazo.

En el contexto de asignación de partidos a horarios, el algoritmo selecciona de manera repetida el horario que maximice la audiencia de un partido en función de los coeficientes de los horarios y la categoría de los equipos, manteniendo las restricciones de los partidos obligatorios y los horarios de coincidencia.

La elección del algoritmo voraz es debido a que es adecuado para este tipo de problema gracias su simplicidad y eficiencia. Aunque no garantiza una solución global óptima, su principal ventaja es que puede generar soluciones razonablemente buenas en un tiempo mucho más corto, ya que no ve todas las posibles combinaciones, en comparación a otras técnicas.